# Contribuciones de Euler, concentración y muestreo por importancia

- **Nivel:** 2, El método en la práctica
- **Requisitos previos:** los cuadernos 01 a 03 de este tema
- **Tiempo de lectura:** unos 45 minutos

## Objetivos

Al terminar este cuaderno puedes:

- decir por qué las contribuciones de Euler de los grupos de una cartera suman su total, y comprobarlo en una simulación: exactamente para el déficit esperado, dentro de una tolerancia para el VaR;
- ver que las contribuciones al VaR son más ruidosas que las del déficit esperado, repitiendo la simulación con distintas semillas;
- comparar las contribuciones de una cartera finita con las asintóticas de forma cerrada, y leer qué dice la diferencia sobre la concentración;
- calcular el índice de Herfindahl y el número efectivo de nombres de una cartera;
- construir un estimador por muestreo por importancia para la cola con el giro exponencial, comprobar que es insesgado, y leer su razón de varianza frente al Monte Carlo simple.

## Preparación

Instala una vez el entorno fijado, en un entorno virtual, con el comando de siempre:

```bash
pip install -r notebooks/requirements.txt
```

o solo los paquetes que usa este cuaderno, con las mismas versiones exactas:

```bash
pip install riskvalidation==1.0.1 numpy==2.4.6 scipy==1.18.1 matplotlib==3.11.2
```

`scripts/setup.sh --notebooks` (`scripts/setup.ps1 -Notebooks` en Windows) construye `.venv-notebooks` por ti. La celda siguiente importa los paquetes e imprime sus versiones.

In [1]:
# Importar las herramientas e imprimir las versiones con que corren, para compararlas con notebooks/requirements.txt
import importlib.metadata as md
import json
import math
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

# El cuaderno está tres carpetas por debajo de la raíz del repositorio
ROOT = Path("../../..").resolve()

for name in ("riskvalidation", "numpy", "scipy", "matplotlib",):
    print(name, md.version(name))

# Mostrar un valor lógico o una etiqueta del motor en español, también dentro de listas y diccionarios
ETIQUETAS = {
    "green": "verde", "amber": "ámbar", "red": "rojo", "not_evaluated": "no evaluada", "illustrative": "ilustrativa",
    "synthetic-calibrated": "sintética calibrada", "withheld-derived-only": "retenidas, solo derivados", "included": "incluidas",
    "fitted": "ajustado", "challenger-not-run": "retador no ejecutado",
}


def si_no(valor):
    if isinstance(valor, dict):
        return {ETIQUETAS.get(k, k): si_no(v) for k, v in valor.items()}
    if getattr(valor, 'ndim', 0) > 0:
        return si_no(valor.tolist())
    if isinstance(valor, (list, tuple)):
        return [si_no(v) for v in valor]
    if isinstance(valor, str):
        return ETIQUETAS.get(valor, valor)
    return "sí" if valor else "no"

riskvalidation 1.0.1
numpy 2.4.6
scipy 1.18.1
matplotlib 3.11.2


## Datos

El archivo comercial e industrial del caso C10, `data/derived/C10/commercial-industrial.json` (propietario: este repositorio; las tasas de pérdida detrás de su ajuste vienen de la tabla `chgallsa` de la Reserva Federal, https://www.federalreserve.gov/releases/chargeoff/chgallsa.htm, dominio público, clase de licencia `mirror-allowed`). Contiene la **cartera ilustrativa** (`outputs.portfolio`: 20 nombres grandes, 100 medianos y 1.000 pequeños, 100 millones de USD, construida para el caso y nunca la de un banco), las contribuciones de Euler versionadas por grupo (`outputs.euler`) y la escalera versionada (`outputs.rungs`, con el peldaño de muestreo por importancia y sus diagnósticos). La celda siguiente lo carga e imprime los grupos y lo que contiene el bloque de Euler versionado.

In [2]:
# Cargar el artefacto, imprimir los grupos y lo que contienen el bloque de Euler versionado y el peldaño de muestreo por importancia
doc = json.loads((ROOT / "data" / "derived" / "C10" / "commercial-industrial.json").read_text(encoding="utf-8"))
out = doc["outputs"]
book = out["portfolio"]
groups = book["groups"]
pd_hat = out["fit"]["maximum_likelihood"]["pd"]["estimate"]
rho_hat = out["fit"]["maximum_likelihood"]["rho"]["estimate"]
rungs = {r["id"]: r for r in out["rungs"]}
euler_block = next(b for b in out["euler"] if b["rung_method"] == "gaussian-copula-one-factor")

print("calculado con riskvalidation", doc["provenance"]["riskvalidation_version"])
for g in groups:
    print(f"{g['id']:9s} {g['n_names']:5d} nombres x {g['ead_each']:>11,.0f} | PD {g['pd']:.5f} | sector {g['sector']}")
print("bloque de Euler versionado:", euler_block["rung_method"], "|", [lv["level"] for lv in euler_block["levels"]], "| VaR método:", euler_block["levels"][0]["var_method"], "| ES método:", euler_block["levels"][0]["es_method"])
print("peldaño de muestreo por importancia versionado:", rungs["c6-is-gaussian"]["simulation"], [(p["name"], round(p["value"], 4)) for p in rungs["c6-is-gaussian"]["parameters"]])

calculado con riskvalidation 1.0.1
g-large      20 nombres x   2,000,000 | PD 0.00506 | sector core
g-mid       100 nombres x     400,000 | PD 0.01012 | sector core
g-small    1000 nombres x      20,000 | PD 0.01517 | sector periphery
bloque de Euler versionado: gaussian-copula-one-factor | [0.99, 0.999] | VaR método: kernel | ES método: tail-sample
peldaño de muestreo por importancia versionado: {'n_scenarios': 100000, 'seed': 4004, 'chunk_size': 16741} [('threshold', 3177000.0), ('factor_shift', -2.3235)]


## Método

### Paso 1: las contribuciones de Euler y por qué suman

Un banco quiere saber de cuánto del capital de la cartera es responsable cada grupo. Para una medida de riesgo $\rho$ positivamente homogénea (escalar todas las exposiciones por $c$ escala el riesgo por $c$), el **principio de Euler** da la contribución del grupo $g$ como la derivada del riesgo respecto de su tamaño, y las contribuciones suman el total (Tasche, 2008, https://arxiv.org/abs/0708.2542). Para las dos medidas del caso son esperanzas condicionales de la pérdida del grupo $L_g$ dado el total $L$:

$$\mathrm{RC}_g^{\mathrm{VaR}}=E\big[L_g\mid L=\mathrm{VaR}_q(L)\big],\qquad \mathrm{RC}_g^{\mathrm{ES}}=E\big[L_g\mid L\ge \mathrm{VaR}_q(L)\big]$$

En una simulación la contribución al ES es un promedio sobre los escenarios de cola, con los pesos de Acerbi y Tasche del cuaderno 03, así que las contribuciones suman exactamente el ES muestral. La contribución al VaR condiciona en un nivel de pérdida, que casi ningún escenario alcanza, así que es ruidosa; el motor la suaviza con el estimador de núcleo de Tasche (2009, https://doi.org/10.1080/14697680802620599), un promedio de Nadaraya-Watson con núcleo gaussiano en el VaR muestral y el ancho de banda de Silverman $1{,}06\,\mathrm{sd}(L)\,S^{-1/5}$ (el ancho de banda es una elección del motor; el artículo no nombra ninguno). La estimación de núcleo tiene sesgo en el borde, así que su suma coincide con el VaR dentro de una tolerancia declarada, no exactamente. `euler_additivity` del motor prueba esa suma: verde cuando la brecha relativa está dentro de la tolerancia, roja sobre ella, sin ámbar. La celda siguiente simula la cartera conservando las pérdidas de cada grupo, calcula ambas contribuciones al 99% y al 99,9%, y prueba sus sumas, con las tolerancias que usa el pipeline: $10^{-6}$ para el ES y 0,01 para el VaR.

In [3]:
# Simular la cartera conservando las pérdidas por grupo, calcular ambas contribuciones y probar que suman
from riskvalidation.portfolio import copula, euler
from riskvalidation.portfolio import portfolio as pf
from riskvalidation.validation.portfolio import euler_additivity

one = pf.portfolio_from_groups(groups, book["currency"], book["lgd_gamma"], one_factor=True)
el = pf.expected_loss(one)
sim = rungs["c2-gaussian-1f"]["simulation"]
sample = copula.simulate_gaussian(one, 100_000, sim["seed"], sim["chunk_size"], track_groups=True)
measures = pf.measures_from_sample(sample.losses, (0.99, 0.999), el)
tolerance = {"var": euler_block["levels"][0]["var_sum_tolerance"], "es": euler_block["levels"][0]["es_sum_tolerance"]}
print("escenarios:", sample.n_scenarios, "| pérdidas por grupo:", sample.group_losses.shape, "| tolerancias:", tolerance)

contributions = {}
for m in measures:
    rc_es = euler.es_contributions(sample.group_losses, sample.losses, m.level)
    rc_var = euler.var_contributions_kernel(sample.group_losses, sample.losses, m.level)
    contributions[m.level] = (rc_var, rc_es)
    test_es = euler_additivity(rc_es, m.es, tolerance["es"], model_id="gaussian-1f", segment=f"es-{m.level}")
    test_var = euler_additivity(rc_var, m.var, tolerance["var"], model_id="gaussian-1f", segment=f"var-{m.level}")
    print(f"q = {m.level}: ES {m.es:,.0f} | suma de las contribuciones al ES {rc_es.sum():,.0f} | brecha relativa {test_es.metric:.1e} | {si_no(test_es.light.value)}")
    print(f"        VaR {m.var:,.0f} | suma de las contribuciones al VaR {rc_var.sum():,.0f} | brecha relativa {test_var.metric:.1e} | {si_no(test_var.light.value)}")

escenarios: 100000 | pérdidas por grupo: (100000, 3) | tolerancias: {'var': 0.01, 'es': 1e-06}
q = 0.99: ES 2,535,057 | suma de las contribuciones al ES 2,535,057 | brecha relativa 1.3e-15 | verde
        VaR 2,088,000 | suma de las contribuciones al VaR 2,087,480 | brecha relativa 2.5e-04 | verde
q = 0.999: ES 3,609,180 | suma de las contribuciones al ES 3,609,180 | brecha relativa 1.3e-16 | verde
        VaR 3,087,000 | suma de las contribuciones al VaR 3,082,159 | brecha relativa 1.6e-03 | verde


Las contribuciones por grupo, como parte del total, dicen a quién pertenece el capital. La celda siguiente imprime, al 99,9%, la participación de exposición, la participación en el VaR y la participación en el ES de cada grupo, y pone al lado las participaciones versionadas de la corrida de un millón de escenarios.

In [4]:
# Imprimir las participaciones de exposición, VaR y ES de cada grupo al 99.9% junto a las versionadas
rc_var_999, rc_es_999 = contributions[0.999]
committed_999 = next(lv for lv in euler_block["levels"] if lv["level"] == 0.999)
exposure = np.array([g["n_names"] * g["ead_each"] for g in groups])
print(f"{'grupo':9s} {'part. exposición':>20s} {'part. VaR':>12s} {'part. ES':>12s} | {'VaR versionado':>18s} {'ES versionado':>16s}")
for k, g in enumerate(groups):
    row = committed_999["rows"][k]
    print(f"{g['id']:9s} {exposure[k] / exposure.sum():20.3f} {rc_var_999[k] / rc_var_999.sum():12.3f} {rc_es_999[k] / rc_es_999.sum():12.3f} | {row['share_var']:18.3f} {row['share_es']:16.3f}")

grupo         part. exposición    part. VaR     part. ES |     VaR versionado    ES versionado
g-large                  0.400        0.591        0.576 |              0.557            0.555
g-mid                    0.400        0.269        0.275 |              0.292            0.292
g-small                  0.200        0.139        0.149 |              0.151            0.153


### Paso 2: el ruido de las contribuciones al VaR

¿Cuánto se mueven las contribuciones de una simulación a la siguiente? La celda siguiente repite la simulación seis veces con 50.000 escenarios y una semilla distinta cada vez, y calcula la participación de cada grupo en el total al 99,9% de tres maneras: la contribución al ES; la contribución al VaR por núcleo; y una estimación simple de la definición, la pérdida media del grupo sobre los escenarios cuyo total está a menos de 2% del VaR (una comprobación de la definición, escrita en la propia celda, no un método del motor). Imprime la participación media de las seis corridas y el coeficiente de variación, la desviación estándar sobre la media.

In [5]:
# Repetir la simulación con seis semillas y comparar la variabilidad de las tres estimaciones de la participación de cada grupo al 99.9%
shares = {"ES": [], "VaR (núcleo)": [], "VaR (banda simple)": []}
for seed in range(900, 906):
    run = copula.simulate_gaussian(one, 50_000, seed, sim["chunk_size"], track_groups=True)
    m = pf.measures_from_sample(run.losses, (0.999,), el)[0]
    rc_e = euler.es_contributions(run.group_losses, run.losses, 0.999)
    rc_v = euler.var_contributions_kernel(run.group_losses, run.losses, 0.999)
    band = np.abs(run.losses - m.var) <= 0.02 * m.var
    plain = run.group_losses[band].mean(axis=0)
    for key, values in zip(shares, (rc_e, rc_v, plain)):
        shares[key].append(values / values.sum())

print(f"{'':20s} " + " ".join(f"{g['id']:>16s}" for g in groups))
for key, runs in shares.items():
    arr = np.array(runs)
    cv = arr.std(axis=0, ddof=1) / arr.mean(axis=0)
    print(f"{key:20s} " + " ".join(f"{arr.mean(axis=0)[k]:7.3f} (cv {cv[k]:.3f})" for k in range(len(groups))))

                              g-large            g-mid          g-small
ES                     0.539 (cv 0.041)   0.304 (cv 0.057)   0.157 (cv 0.047)
VaR (núcleo)           0.557 (cv 0.054)   0.284 (cv 0.082)   0.159 (cv 0.067)
VaR (banda simple)     0.552 (cv 0.061)   0.289 (cv 0.088)   0.159 (cv 0.076)


### Paso 3: las contribuciones asintóticas y la concentración

Para una cartera de un factor infinitamente granular la contribución al VaR del nombre $i$ tiene forma cerrada: la pérdida es monótona en el factor, así que el VaR es la pérdida media condicional en el cuantil del factor $z=\Phi^{-1}(1-q)$, y $\mathrm{RC}_i=\mathrm{EAD}_i\,\mathrm{LGD}_i\,p_i(z)$ con $p_i$ la PD condicional del cuaderno 01 (Emmer y Tasche, 2005, https://arxiv.org/abs/cond-mat/0302402; `asrf_var_contributions` del motor declara que este es el término principal, invariante a la cartera, y no su contribución completa, que agrega términos de granularidad). Estas contribuciones dependen del nombre solo por su propia PD y exposición, sea cual sea el resto de la cartera. Una cartera finita se aparta de ellas por la concentración. El motor mide la concentración con el índice de Herfindahl $\mathrm{HHI}=\sum_i s_i^2$ de las participaciones de exposición y el número efectivo de nombres $1/\mathrm{HHI}$, que vale $N$ para $N$ nombres iguales y 1 para un solo nombre. La celda siguiente calcula las contribuciones asintóticas por grupo, las compara con las simuladas, e imprime la concentración de la cartera y de la misma cartera con exposiciones iguales.

In [6]:
# Calcular las contribuciones asintóticas por grupo, compararlas con las simuladas e imprimir la concentración
cut = np.cumsum([0] + [g["n_names"] for g in groups])
asymptotic = euler.asrf_var_contributions(one.ead, one.pd, one.lgd, rho_hat, 0.999)
by_group = np.array([asymptotic[a:b].sum() for a, b in zip(cut[:-1], cut[1:])])
anchor = committed_999["anchor"]
print("VaR asintótico al 99.9%:", f"{asymptotic.sum():,.0f}", "| versionado:", f"{anchor['asrf_var']:,.0f}")
print(f"{'grupo':9s} {'asintótica':>14s} {'participación':>12s} {'participación simulada':>24s}")
for k, g in enumerate(groups):
    print(f"{g['id']:9s} {by_group[k]:14,.0f} {by_group[k] / by_group.sum():12.3f} {rc_var_999[k] / rc_var_999.sum():24.3f}")

print("HHI:", euler.hhi(one.ead), "| nombres efectivos:", round(euler.effective_names(one.ead), 2), "| versionado:", book["hhi"], round(book["effective_names"], 2))
equal = np.full(one.ead.size, one.ead.sum() / one.ead.size)
print("mismo total, exposiciones iguales: HHI", round(euler.hhi(equal), 6), "| nombres efectivos", round(euler.effective_names(equal), 1), "| participación del mayor nombre", book["largest_share"])

VaR asintótico al 99.9%: 2,259,748 | versionado: 2,259,748
grupo         asintótica participación   participación simulada
g-large          576,288        0.255                    0.591
g-mid            998,871        0.442                    0.269
g-small          684,590        0.303                    0.139
HHI: 0.00964 | nombres efectivos: 103.73 | versionado: 0.00964 103.73
mismo total, exposiciones iguales: HHI 0.000893 | nombres efectivos 1120.0 | participación del mayor nombre 0.02


### Paso 4: el muestreo por importancia para la cola

Para estimar un cuantil al 99,9% con Monte Carlo simple, solo el 0,1% de los escenarios cae en la cola, y los errores de medias por lotes del cuaderno 03 fueron grandes. El **muestreo por importancia** sortea los escenarios desde una distribución que visita la cola más seguido y corrige cada uno con una razón de verosimilitud. Para la cópula gaussiana con factores no correlacionados, el giro exponencial de Glasserman y Li (2005, https://doi.org/10.1287/mnsc.1050.0415) funciona en dos partes. Dado el factor sistemático $z$, cada probabilidad de incumplimiento condicional $p_i(z)$ se inclina, con $c_i=\mathrm{EAD}_i\,\mathrm{LGD}_i$ la pérdida del nombre $i$:

$$p_{i,\theta}(z)=\frac{p_i(z)\,e^{\theta c_i}}{1+p_i(z)\,(e^{\theta c_i}-1)},\qquad \theta=\theta(z)\ \text{la raíz de}\ \sum_i c_i\,p_{i,\theta}(z)=x$$

de modo que la pérdida media condicional inclinada es igual al umbral $x$ (y $\theta=0$ donde la media sin inclinar ya lo supera); luego el propio factor se sortea con una media desplazada $\mu$. Cada escenario lleva el peso $w=\exp\!\big(-\theta L+\sum_i\ln(1+p_i(e^{\theta c_i}-1))\big)\exp\!\big(-\mu z+\mu^2/2\big)$ en el caso de un factor, con $E_Q[w]=1$ y $E_Q[wL]=\mathrm{EL}$ para cualquier $\theta(z)\ge0$ y cualquier $\mu$: el estimador es insesgado sea cual sea la inclinación, y solo su varianza depende de ella. Una advertencia de la investigación: el dossier marca su transcripción del giro y del desplazamiento de la media como UNVERIFIED frente al artículo, que es de pago, así que el motor no acepta nada por confianza y su aceptación es la insesgadez comprobada frente al Monte Carlo simple, que repetimos abajo. La celda siguiente toma un escenario, el valor del factor que usó el peldaño versionado para su desplazamiento, y muestra el giro en acción: las probabilidades de incumplimiento condicionales antes y después, y la comprobación de que la pérdida media inclinada es igual al objetivo. El objetivo es el VaR al 99,9% versionado de Monte Carlo simple.

In [7]:
# Mostrar el giro exponencial para un valor del factor: probabilidades de incumplimiento condicionales antes y después, y la pérdida media inclinada
from riskvalidation.engines.vasicek import conditional_pd
from riskvalidation.portfolio import importance

threshold = rungs["c2-gaussian-1f"]["measures"][1]["var"]
shift = next(p["value"] for p in rungs["c6-is-gaussian"]["parameters"] if p["name"] == "factor_shift")
loss_of_name = one.ead * one.lgd
p_cond = np.array([float(conditional_pd(g["pd"], rho_hat, shift)) for g in groups for _ in range(g["n_names"])])
theta = importance.solve_theta(p_cond, loss_of_name, threshold)
p_twist = importance.twisted_pd(p_cond, loss_of_name, theta)

print("pérdida objetivo (VaR de Monte Carlo simple versionado al 99.9%):", f"{threshold:,.0f}", "| valor del factor:", shift)
print("pérdida media dado el factor, sin inclinar:", f"{loss_of_name @ p_cond:,.0f}", "| theta:", f"{theta:.3e}")
print("pérdida media dado el factor, inclinada:", f"{loss_of_name @ p_twist:,.0f}")
for k, g in enumerate(groups):
    first = cut[k]
    print(f"{g['id']:9s} PD condicional antes {p_cond[first]:.4f} | después del giro {p_twist[first]:.4f}")

pérdida objetivo (VaR de Monte Carlo simple versionado al 99.9%): 3,177,000 | valor del factor: -2.32351367
pérdida media dado el factor, sin inclinar: 1,479,392 | theta: 1.879e-06
pérdida media dado el factor, inclinada: 3,177,000
g-large   PD condicional antes 0.0201 | después del giro 0.1001
g-mid     PD condicional antes 0.0364 | después del giro 0.0503
g-small   PD condicional antes 0.0513 | después del giro 0.0522


El giro empuja más a los nombres grandes, porque llevan las pérdidas mayores. Ahora el estimador. La celda siguiente sortea 20.000 escenarios de muestreo por importancia con la semilla y el objetivo del pipeline (una quinta parte de la corrida versionada), y comprueba lo que promete la teoría: el desplazamiento del factor que resolvió el motor, la media de los pesos (1 en esperanza) y la pérdida media ponderada (la pérdida esperada en esperanza). Luego lee el VaR y el ES al 99% y al 99,9% con `weighted_var_es`, que normaliza la cola por el número de escenarios y nunca por la suma realizada de los pesos, y los pone junto al Monte Carlo simple de los 100.000 escenarios.

In [8]:
# Sortear 20,000 escenarios de muestreo por importancia, comprobar las identidades de insesgadez y leer VaR y ES junto al Monte Carlo simple
n_is = 20_000
result = importance.importance_sample_gaussian(one, threshold, n_is, rungs["c6-is-gaussian"]["simulation"]["seed"], None, sim["chunk_size"])
w = result.weights
print("desplazamiento del factor resuelto por el motor:", round(float(result.factor_shift[0]), 6), "| versionado:", shift)
print("media de los pesos (1 en esperanza):", round(float(w.mean()), 4), "| pérdida media ponderada:", f"{float(np.mean(w * result.losses)):,.0f}", "| EL:", f"{el:,.0f}")
print("participación de muestra efectiva:", round(float(w.sum() ** 2 / (w.size * np.sum(w * w))), 5), "| versionada (100,000 escenarios):", round(next(d["value"] for d in rungs["c6-is-gaussian"]["diagnostics"] if d["name"] == "effective_sample_share"), 6))

plain = {m.level: m for m in measures}
committed_is = {m["level"]: m for m in rungs["c6-is-gaussian"]["measures"]}
print(f"{'q':>6s} {'VaR IS':>12s} {'error est.':>10s} {'VaR simple (100k)':>22s} {'error est.':>10s} | {'ES IS':>12s} {'error est.':>10s} {'ES simple':>12s}")
is_se = {}
for q in (0.99, 0.999):
    var_i, es_i, se_v, se_e = importance.weighted_var_es(result, q)
    is_se[q] = (se_v, se_e)
    print(f"{q:6.3f} {var_i:12,.0f} {se_v:10,.0f} {plain[q].var:22,.0f} {plain[q].var_se:10,.0f} | {es_i:12,.0f} {se_e:10,.0f} {plain[q].es:12,.0f}")
print("IS versionado (100,000 escenarios): VaR", f"{committed_is[0.999]['var']:,.0f}", "| ES", f"{committed_is[0.999]['es']:,.0f}", "| errores", round(committed_is[0.999]["var_se"]), round(committed_is[0.999]["es_se"]))

desplazamiento del factor resuelto por el motor: -2.323514 | versionado: -2.32351367
media de los pesos (1 en esperanza): 0.5665 | pérdida media ponderada: 338,187 | EL: 409,721
participación de muestra efectiva: 0.00202 | versionada (100,000 escenarios): 0.000167
     q       VaR IS error est.      VaR simple (100k) error est. |        ES IS error est.    ES simple
 0.990    2,106,000      7,711              2,088,000     18,433 |    2,578,515      3,577    2,535,057
 0.999    3,204,000      5,368              3,087,000     54,340 |    3,702,227      5,042    3,609,180
IS versionado (100,000 escenarios): VaR 3,204,000 | ES 3,710,983 | errores 2432 2697


La razón de varianza es el número de escenarios de Monte Carlo simple que vale un escenario de muestreo por importancia. `variance_ratio` del motor es $(\mathrm{ee}_{\text{simple}}/\mathrm{ee}_{\text{IS}})^2$ con igual tamaño de muestra, así que el error simple de 100.000 escenarios se escala primero a 20.000 por $\sqrt{100.000/20.000}$. La celda siguiente la calcula para el VaR y el ES al 99,9% y pone al lado las razones versionadas de la corrida de 100.000 escenarios.

In [9]:
# Calcular la razón de varianza frente al Monte Carlo simple con igual tamaño de muestra y compararla con las razones versionadas
scale = math.sqrt(100_000 / n_is)
ratio_var = importance.variance_ratio(plain[0.999].var_se * scale, is_se[0.999][0])
ratio_es = importance.variance_ratio(plain[0.999].es_se * scale, is_se[0.999][1])
diag = {d["name"]: d["value"] for d in rungs["c6-is-gaussian"]["diagnostics"]}
print("razón de varianza, VaR 99.9%:", round(ratio_var), "| versionada:", round(diag["variance_ratio_vs_plain_mc"]))
print("razón de varianza, ES 99.9%:", round(ratio_es), "| versionada:", round(diag["variance_ratio_es_vs_plain_mc"]))
print("cada escenario de muestreo por importancia vale aproximadamente", round(ratio_var), "escenarios simples para el VaR")

razón de varianza, VaR 99.9%: 512 | versionada: 389
razón de varianza, ES 99.9%: 1500 | versionada: 644
cada escenario de muestreo por importancia vale aproximadamente 512 escenarios simples para el VaR


## Resultados

- **Las contribuciones suman, exactamente para el ES y dentro de la tolerancia para el VaR.** Las contribuciones al ES de los tres grupos suman el ES muestral con una brecha relativa de 1,3e-15 al 99% y 1,3e-16 al 99,9%, ambas en verde: los pesos de Acerbi y Tasche hacen de la suma una identidad. Las contribuciones al VaR por núcleo suman 2.087.480 frente a un VaR de 2.088.000 al 99% (brecha 2,5e-4) y 3.082.159 frente a 3.087.000 al 99,9% (brecha 1,6e-3), ambas dentro de la tolerancia de 0,01 y en verde. Una brecha bajo la tolerancia no es cero, y la prueba es una puerta de Contraste, no un umbral regulatorio.
- **Los nombres grandes llevan mucho más capital que su exposición.** Los 20 nombres grandes son el 40% de la exposición y llevan el 59,1% del VaR al 99,9% y el 57,6% del ES en esta muestra de 100.000 escenarios (la corrida versionada de un millón dice 55,7% y 55,5%). Los nombres medianos son el 40% de la exposición y llevan el 26,9% del VaR, los 1.000 nombres pequeños el 20% de la exposición y el 13,9%.
- **Las contribuciones al VaR son más ruidosas que las del ES, aunque con 50.000 escenarios no por mucho.** Entre seis semillas el coeficiente de variación de la participación al 99,9% de los nombres grandes es 4,1% para la contribución al ES, 5,4% para la contribución al VaR por núcleo y 6,1% para la estimación simple con escenarios cercanos al VaR, y el mismo orden se mantiene para los nombres medianos y pequeños (5,7%, 8,2%, 8,8% y 4,7%, 6,7%, 7,6%). El núcleo suaviza la estimación simple, y la contribución al ES es más estable que ambas. La diferencia entre las participaciones en el VaR de la corrida única y de la versionada (0,591 y 0,557 para los nombres grandes) es el mismo ruido en acción.
- **Las contribuciones asintóticas no ven la concentración.** Las contribuciones ASRF de forma cerrada suman 2.259.748 USD al 99,9%, iguales al valor versionado, y dan a los nombres grandes el 25,5% del capital frente al 59,1% simulado, y a los nombres medianos el 44,2% frente al 26,9%. Dependen de cada nombre solo por su propia PD y exposición, así que son invariantes a la cartera: un nombre de 2 millones de USD recibe el mismo capital por dólar en una cartera de 1.120 nombres y en una de 20. La cartera finita concentra el capital en sus nombres mayores, y una forma cerrada que ignora el resto de la cartera no lo ve.
- **La concentración en un número.** El índice de Herfindahl es 0,00964 y el número efectivo de nombres 103,73 de los 1.120, los valores versionados; el mismo total repartido en exposiciones iguales sería 0,000893 y 1.120 nombres efectivos. El mayor nombre es el 2% de la exposición.
- **El giro exponencial lleva la pérdida adonde está la cola.** Para el valor del factor del desplazamiento versionado, -2,32, la pérdida media dado el factor es 1.479.392 USD sin inclinar y, tras resolver theta (1,879e-6), 3.177.000 USD, el objetivo. El giro sube la probabilidad de incumplimiento condicional de los nombres grandes de 2,01% a 10,01%, la de los medianos de 3,64% a 5,03% y la de los pequeños de 5,13% a 5,22%: empuja donde la pérdida por nombre es mayor.
- **El estimador es insesgado en esperanza, y la muestra muestra qué significa eso.** El desplazamiento del factor del motor es igual al versionado. Con 20.000 escenarios los pesos promedian 0,5665, no 1, y la pérdida media ponderada es 338.187 USD frente a la pérdida esperada de 409.721 USD: los pesos tienen cola pesada y una muestra de 20.000 está lejos de su media (la participación de muestra efectiva es 0,2%). Esta es la razón por la que `weighted_var_es` normaliza la cola por el número de escenarios y nunca por la suma realizada de los pesos. El VaR al 99,9% es 3.204.000 USD con un error estándar de 5.368, igual al de la corrida versionada de muestreo por importancia, y a unos dos errores estándar de la corrida simple de un millón de escenarios, 3.177.000 ± 15.163; el Monte Carlo simple con 100.000 escenarios dio 3.087.000 ± 54.340. El ES es 3.702.227 ± 5.042 frente a 3.683.187 ± 21.634 de la corrida simple versionada. La prueba del giro es esta concordancia, porque el dossier marca la transcripción del giro como no verificada frente al artículo.
- **La ganancia es grande.** Cada escenario de muestreo por importancia vale unos 512 escenarios simples para el VaR al 99,9% y unos 1.500 para el ES con igual tamaño de muestra, frente a 389 y 644 en la corrida versionada de 100.000 escenarios. Las razones difieren entre corridas porque ambos errores son estimaciones por medias de lotes y los errores de cola son ruidosos en sí mismos.

Lo que este cuaderno no muestra: el muestreo por importancia para la t de Student (el motor no tiene giro para ella, y el peldaño versionado registra que no corrió), el giro multifactor (el motor soporta solo factores no correlacionados), ni las contribuciones de nombres individuales en vez de grupos.

## Resumen

- Las contribuciones de Euler reparten el capital entre grupos y suman el total: exactamente para el déficit esperado (`es_contributions`), dentro de una tolerancia declarada para el VaR (`var_contributions_kernel`), comprobado por `euler_additivity`.
- Las contribuciones al VaR son las más ruidosas de las dos; el núcleo las suaviza pero siguen más ruidosas que las del ES.
- Las contribuciones asintóticas (`asrf_var_contributions`) son invariantes a la cartera y no ven la concentración: los nombres grandes llevan mucho más capital en la cartera finita que el que les da la forma cerrada. `hhi` y `effective_names` miden la concentración.
- El muestreo por importancia con el giro exponencial (`solve_theta`, `twisted_pd`, `importance_sample_gaussian`, `weighted_var_es`) alcanza la cola al 99,9% con cientos de veces menos varianza, y su insesgadez es la prueba de aceptación mientras la transcripción del giro no esté verificada.

**Qué probar a continuación.** Cambia las semillas del Paso 2 y mira cómo se mantiene el orden de los tres estimadores. Luego abre el cuaderno 06, donde las pruebas de esta batería se leen juntas en los cinco tipos de préstamo.

## Referencias

- Tasche, D. (2008), Capital allocation to business units and sub-portfolios: the Euler principle, arXiv 0708.2542. https://arxiv.org/abs/0708.2542
- Tasche, D. (2009), Capital allocation for credit portfolios with kernel estimators, Quantitative Finance 9(5):581-595. https://doi.org/10.1080/14697680802620599
- Emmer, S. y Tasche, D. (2005), Calculating credit risk capital charges with the one-factor model, Journal of Risk 7(2):85-101, arXiv cond-mat/0302402. https://arxiv.org/abs/cond-mat/0302402
- Glasserman, P. y Li, J. (2005), Importance sampling for portfolio credit risk, Management Science 51(11):1643-1656. https://doi.org/10.1287/mnsc.1050.0415
- Acerbi, C. y Tasche, D. (2002), On the coherence of expected shortfall, Journal of Banking and Finance 26(7):1487-1503. https://doi.org/10.1016/S0378-4266(02)00283-2
- Federal Reserve charge-off table `chgallsa`, https://www.federalreserve.gov/releases/chargeoff/chgallsa.htm (public domain; licence class mirror-allowed).
- Persisted research dossier 04 of this product (credit-risk models, state of the art), sections H.2, H.6 and H.7 (Euler contributions, concentration and rare-event simulation; the transcription of the twist is marked UNVERIFIED there).
- The engine's contract for these functions: `docs/contracts/engine/` of this repository.